In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

from cobolt.utils import SingleData, MultiomicDataset
from cobolt.model import Cobolt
import os
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score as nmi_score
from time import time
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, adjusted_mutual_info_score
from sklearn.metrics import fowlkes_mallows_score, homogeneity_score, completeness_score, v_measure_score

/home/chengyue/anaconda3/envs/cobolt/lib/python3.9/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/chengyue/anaconda3/envs/cobolt/lib/python3.9/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/chengyue/anaconda3/envs/cobolt/lib/python3.9/site-packages/uma

In [2]:
def eva(y_true, y_pred):
    nmi = nmi_score(y_true, y_pred)
    ari = ari_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    fmi = fowlkes_mallows_score(y_true, y_pred)
    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v = v_measure_score(y_true, y_pred)
    return nmi, ari, ami, fmi, hom, com, v



In [ ]:
import scanpy as sc
rna = sc.read_h5ad('./human_pbmc_3k_rna.h5ad')
count_rna = rna.X
rna.obs['barcodes'] = rna.obs_names#['orig.ident'].index
barcodes_rna = rna.obs['barcodes']
features_rna = rna.var_names#['features'].index#pbmc10kpublic is feature feature_index
snare_mrna = SingleData("GeneExpr", "SNARE-seq", features_rna, count_rna, barcodes_rna)
atac = sc.read_h5ad('./human_pbmc_3k_atac.h5ad')
count_atac = atac.X
atac.obs['barcodes'] = atac.obs_names#['orig.ident'].index
barcodes_atac = atac.obs['barcodes']
features_atac = atac.var_names#['features'].index
snare_atac = SingleData("ChromAccess", "SNARE-seq", features_atac, count_atac, barcodes_atac)

In [ ]:
multi_dt = MultiomicDataset.from_singledata(snare_atac, snare_mrna)
model = Cobolt(dataset=multi_dt, n_latent=10)
model.train(num_epochs=10)

model.calc_all_latent()
latent = model.get_all_latent()
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

cell_name = pd.read_csv('/home/chengyue/data/multi-omics/human_pbmc_3k_label_a.csv', usecols=[1])
cell_type, y = np.unique(cell_name, return_inverse=True)
n_clusters = int(max(y) - min(y) + 1)  

kmeans = KMeans(n_clusters = n_clusters, n_init=20)
latent = latent[0]

y_pred_z = kmeans.fit_predict(latent) 

nmi, ari, ami, fmi, hom, com, v = eva(y, y_pred_z)
print('z for clustering, NMI:{:.4f}, ARI:{:.4f}, AMI:{:.4f}, FMI:{:.4f}, HOM:{:.4f}, COM:{:.4f}, V:{:.4f}'.format(nmi, ari, ami, fmi, hom, com, v))

100%|██████████| 10/10 [01:50<00:00, 11.02s/it]


z for clustering, NMI:0.5128, ARI:0.3456, AMI:0.5105, FMI:0.4410, HOM:0.5013, COM:0.5247, V:0.5128
